<a href="https://colab.research.google.com/github/Rimshakalhoro/flyrank-ml-internship-rimsha/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Rimshakalhoro/flyrank-ml-internship-rimsha/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

### Unit of analysis

The unit of analysis for my provisional lane is one content page.

Each row in the starter dataset represents one anonymized content item identified by `content_id`. The purpose is to compare pages using observable search-performance and content signals and prioritize pages that may deserve human review.

### Time window

The dataset contains aggregated 90-day performance fields, including impressions, clicks, pageviews, sessions, users, and engagement-related measures.

It also contains `last_30d` and `prev_30d` fields for impressions, clicks, and sessions. These windows can be used to measure directional changes between two observed 30-day periods.

The dataset does not provide the exact calendar start and end dates in the fields shown here, so I will describe the available periods as relative windows rather than claiming specific dates.

In [9]:
import pandas as pd

# Load the starter dataset from your GitHub repository
url = "https://raw.githubusercontent.com/Rimshakalhoro/flyrank-ml-internship-rimsha/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(url)

print("Rows:", len(df))
print("Columns:", len(df.columns))

print("\nColumn names:")
print(df.columns.tolist())

df.head()


Rows: 30000
Columns: 44

Column names:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


### Feature fields

My candidate feature fields are observable search, content, freshness, and engagement signals that could be available when prioritizing a page for review.

Search and performance features include:

`search_volume`, `competition`, `competition_level`, `cpc`, `impressions_90d`, `clicks_90d`, `pageviews_90d`, `sessions_90d`, `users_90d`, `engaged_sessions_90d`, `ai_sessions_90d`, `scroll_events_90d`, `days_with_impressions`, and `days_with_sessions`.

Content and freshness features include:

`content_type`, `main_intent`, `word_count`, `char_count`, `content_age_days`, `age_tier`, `age_tier_order`, `days_since_last_update`, `freshness_tier`, `word_count_tier`, and `char_count_tier`.

Derived performance signals include:

`ctr`, `avg_position`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct`, `impression_tier`, and `position_tier`.

### Label / proxy

My provisional proxy is `trend_direction`, which describes the observed direction of performance change in the available dataset.

For a binary version of the proxy, I can define a declining indicator where:

`is_declining = 1` when `trend_direction` is `"down"`.

This is a proxy for prioritization and decision-support. It does not prove that a page should be refreshed or that refreshing it will improve performance.

### Context fields

`content_id` and `client_id` are context and identifier fields. They help identify or group records but should not be used as predictive features.

### Excluded fields

I exclude `content_id` and `client_id` from model features because identifiers can create meaningless patterns and may cause a model to memorize records or groups instead of learning useful relationships.

I also treat `trend_pct` carefully because it is directly related to the observed trend outcome and may create leakage depending on how the final target is defined.

Any feature that would not be available at the time of a real recommendation will also be excluded from the final model.

In [10]:
# ==========================================
# SECTION 2 — FIELD CATEGORIES
# ==========================================

context_fields = [
    "content_id",
    "client_id"
]

label_fields = [
    "trend_direction"
]

excluded_fields = [
    "content_id",
    "client_id",
    "trend_pct"
]

feature_fields = [
    col for col in df.columns
    if col not in excluded_fields
    and col not in label_fields
    and col not in context_fields
]

print("FEATURE FIELDS:")
print(feature_fields)

print("\nLABEL / PROXY:")
print(label_fields)

print("\nCONTEXT FIELDS:")
print(context_fields)

print("\nEXCLUDED FIELDS:")
print(excluded_fields)

print("\nNumber of candidate features:", len(feature_fields))


FEATURE FIELDS:
['search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier']

LABEL / PROXY:
['trend_direction']

CONTEXT FIELDS:
['content_id', 'client_id']

EXCLUDED FIELDS:
['content_id', 'client_id', 'trend_pct']

Number of candidate features: 40


### Verification queries

I verified the data contract with direct dataframe checks rather than relying only on assumptions.

The checks below examine:

1. The number of rows.
2. Whether the content identifier is unique.
3. Missing values in the available fields.
4. The observed window-related fields.
5. Whether the dataframe matches the intended page-level unit of analysis.

In [11]:
# ==========================================
# SECTION 3 — VERIFY THE DATA CONTRACT
# ==========================================

print("TOTAL ROWS:", len(df))
print("TOTAL COLUMNS:", len(df.columns))

# Verify the unit of analysis
print("\nUNIQUE CONTENT IDs:")
print(df["content_id"].nunique())

print("\nDUPLICATE CONTENT IDs:")
print(df.duplicated(subset=["content_id"]).sum())

# Missing values
print("\nMISSING VALUES:")

missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)

if len(missing) == 0:
    print("No missing values found.")
else:
    print(missing)

# Verify observed windows
window_columns = [
    col for col in df.columns
    if "90d" in col.lower()
    or "30d" in col.lower()
    or "trend" in col.lower()
]

print("\nOBSERVED WINDOW / TREND FIELDS:")
for col in window_columns:
    print("-", col)

TOTAL ROWS: 30000
TOTAL COLUMNS: 44

UNIQUE CONTENT IDs:
30000

DUPLICATE CONTENT IDs:
0

MISSING VALUES:
provider_used        21438
word_count_tier       7699
char_count            7699
word_count            7699
char_count_tier       7699
model_used            5733
trend_pct             3388
competition_level     2610
search_volume         2468
competition           2468
cpc                   2468
main_intent           2374
scroll_rate            125
dtype: int64

OBSERVED WINDOW / TREND FIELDS:
- impressions_90d
- clicks_90d
- pageviews_90d
- sessions_90d
- users_90d
- engaged_sessions_90d
- ai_sessions_90d
- scroll_events_90d
- impressions_last_30d
- clicks_last_30d
- sessions_last_30d
- impressions_prev_30d
- clicks_prev_30d
- sessions_prev_30d
- trend_direction
- trend_pct


### Data limits

This dataset is observational, so it can show measured patterns and associations but cannot by itself prove that one action caused another outcome.

The available dataset may also use aggregated time windows rather than providing a complete daily history for every page. Therefore, overlapping windows and timing must be considered carefully when defining features and labels.

The data can support decision-making about which pages may deserve review, but it cannot prove that refreshing a page will improve its search performance.

A page-level score should therefore be treated as decision-support rather than an automatic instruction.

The dataset is anonymized, which protects privacy but limits the ability to interpret page topics, client context, or business goals.

Any missing fields or differences in data availability will be measured and documented rather than assumed to mean poor performance.

In [12]:
print("DATA LIMIT CHECK")

print("\nTotal rows:", len(df))
print("Total columns:", len(df.columns))

print("\nColumns with missing values:")

missing_count = (df.isnull().sum() > 0).sum()

print(missing_count)

print("\nWindow-related fields:")

for col in df.columns:
    if any(x in col.lower() for x in ["90d", "60d", "30d", "trend"]):
        print("-", col)


DATA LIMIT CHECK

Total rows: 30000
Total columns: 44

Columns with missing values:
13

Window-related fields:
- impressions_90d
- clicks_90d
- pageviews_90d
- sessions_90d
- users_90d
- engaged_sessions_90d
- ai_sessions_90d
- scroll_events_90d
- impressions_last_30d
- clicks_last_30d
- sessions_last_30d
- impressions_prev_30d
- clicks_prev_30d
- sessions_prev_30d
- trend_direction
- trend_pct


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.